In [0]:
filename = dbutils.widgets.get("filename")
# filename = "orders_new.csv"
# fnameWithoutExt = filename.split(".")[0]
print(filename)

In [0]:
path = "abfss://sales@trendytechstracc.dfs.core.windows.net"
landing_folder = path + "/landing"
staging_folder = path + "/staging"
discarded_folder = path + "/discarded" 
order_items_folder = path + "/order_items"

In [0]:
ordersDf = spark.read.csv(f'{landing_folder}/{filename}', inferSchema=True, header=True)
display(ordersDf)

### 1 st Condition : move the file to discarded folder if it has duplicate order_id's

In [0]:
errorFlag = False

ordersCount = ordersDf.count()
print(ordersCount)

distinctOrdersCount = ordersDf.select('order_id').distinct().count()
print(distinctOrdersCount)

if ordersCount != distinctOrdersCount:
    errorFlag = True

if errorFlag:
    dbutils.fs.mv(f'{landing_folder}/{filename}', discarded_folder)
    dbutils.notebook.exit('{"errorFlg": "true", "errorMsg": "Orderid is repeated"}')

ordersDf.createOrReplaceTempView("orders")


In [0]:
spark.sql("select * from orders")

*** NOTE: we can't use * (regex) here like `{landing_folder}/*` this will just bring all the files, its better to work file by file basis. In the event when we are moving the files if any new files comes it also get moved to staging/discarded without any check and thats a bug

### 2nd Condition : move the file to discarded folder if order status doesn't match the ones stored in our sql database

#### setup database connection to lookup valid order status

In [0]:
dbServer = 'trendytechsqlserv'
dbPort = '1433'
dbName = 'free-sql-db-5673187'
dbUser = 'tt-sql-user'

connectionUrl = f'jdbc:sqlserver://{dbServer}.database.windows.net:{dbPort};database={dbName};user={dbUser};'

dbPassword =dbutils.secrets.get(scope = 'salesprojectscope', key='sql-password')

connectionProperties = {
'password': dbPassword,
'driver':'com.microsoft.sqlserver.jdbc.SQLServerDriver'
}

**only possible from interactive cluster or gen purp compute dont user serverless (sometimes works some times doesn't)**

In [0]:
validStatusDf = spark.read.jdbc(url=connectionUrl, table='dbo.valid_order_status', properties= connectionProperties)

In [0]:
display(validStatusDf)

In [0]:
validStatusDf.createOrReplaceTempView("valid_status")

In [0]:
invalidRowsDf = spark.sql("select * from orders where order_status not in (select * from valid_status)")

In [0]:
display(invalidRowsDf)

In [0]:
if invalidRowsDf.count() > 0:
    errorFlag = True 

if errorFlag:
    dbutils.fs.mv(f'{landing_folder}/{filename}', discarded_folder)
    dbutils.notebook.exit('{"errorFlg": "true", "errorMsg": "Invalid order status found"}')
else:
    dbutils.fs.mv(f'{landing_folder}/{filename}', staging_folder)
    print('{"errorFlg": "false", "errorMsg": "All good"}')
    # recreate DF and table as files moved from /landing to /staging location
    ordersDf = spark.read.csv(f'{staging_folder}/{filename}', inferSchema=True, header=True)
    ordersDf.createOrReplaceTempView("orders")

### Read Order Items data

In [0]:
orderItemsDf = spark.read.csv(order_items_folder, header=True, inferSchema=True)
display(orderItemsDf)

In [0]:
orderItemsDf.createOrReplaceTempView("order_items")

### Read Customers Data

In [0]:
customerDf = spark.read.jdbc(url=connectionUrl, table='dbo.customers', properties= connectionProperties)

display(customerDf)

In [0]:
customerDf.createOrReplaceTempView("customers")

## Join the Customers, Orders and OrderItems table for analytics

In [0]:
resultDf1 = spark.sql("""
select 
    c.customer_id, 
    c.customer_fname, 
    c.customer_lname, 
    c.customer_city, 
    c.customer_state, 
    c.customer_zipcode, 
    count(distinct o.order_id) as num_orders_placed, 
    round(sum(oi.order_item_subtotal), 2) as total_amount 
from customers c
join orders o on c.customer_id = o.customer_id
join order_items oi on o.order_id = oi.order_item_order_id
group by 
    c.customer_id, c.customer_fname, c.customer_lname, 
    c.customer_city, c.customer_state, c.customer_zipcode 
order by total_amount desc""")

display(resultDf1)

### Put the results of join to azure sql db for reporting

In [0]:
resultDf1.write.jdbc(url=connectionUrl, table='dbo.sales_reporting', properties= connectionProperties, mode='overwrite')